## Import libraries

In [14]:
from pathlib import Path
import json
import numpy as np

from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct

## Project Paths

In [15]:
PROJECT_PATH = Path.cwd().parent

CHUNKS_FILE = (PROJECT_PATH / "data" / "processed" / "chunks" / "chunks.jsonl")

EMBEDDINGS_FILE = (PROJECT_PATH / "data" / "processed" / "embeddings" / "embeddings.npy")

QDRANT_PATH = (PROJECT_PATH / "data" / "processed" / "vector_store" / "qdrant")

COLLECTION_NAME = "financial_policies"

## Load Chunks and Embeddings

In [18]:
with open(CHUNKS_FILE, 'r', encoding='utf-8') as f:
    chunks = [json.loads(line) for line in f]

embeddings = np.load(EMBEDDINGS_FILE)

print("Number of Chunks:", len(chunks))
print("Embedding Shape:", embeddings.shape)
print("Embedding dtype:", embeddings.dtype)

Number of Chunks: 295
Embedding Shape: (295, 1024)
Embedding dtype: float32


## Validate Chunk-Embedding Alignment

In [19]:
assert len(chunks) == len(embeddings), (
    f"Mismatch: {len(chunks)} chunks vs {len(embeddings)} embeddings"
)

assert embeddings.shape[1] == 1024
print("Chunk and embedding alignment validate")

Chunk and embedding alignment validate



## Initialize Qdrant

In [20]:
QDRANT_PATH.mkdir(parents=True, exist_ok=True)

client = QdrantClient(path=str(QDRANT_PATH))

print("Qdrant client initialized.")
print("Storage path:", QDRANT_PATH)

Qdrant client initialized.
Storage path: /Users/pushkarkamat/Desktop/financial-rag/data/processed/vector_store/qdrant


## Create Qdrant Collection

In [22]:
existing_collecting = [
    collection.name
    for collection in client .get_collections().collections
]

if COLLECTION_NAME in existing_collecting:
    client.delete_collection(COLLECTION_NAME)

client.create_collection(
    collection_name=COLLECTION_NAME,
    vectors_config=VectorParams(
        size=embeddings.shape[1],
        distance=Distance.COSINE,
    ),
)

True

## Prepare Qdrant Points

In [25]:
points = []

for i, (chunk, embedding) in enumerate(zip(chunks, embeddings)):
    payload = {
        "text": chunk["page_content"],
        **chunk["metadata"],
    }

    points.append(
        PointStruct(
            id=i,
            vector=embeddings.tolist(),
            payload=payload
        )
    )

print("Points prepared:", len(points))

Points prepared: 295


## Upload Vectors to Qdrant

In [26]:
client.upsert(
    collection_name=COLLECTION_NAME,
    points=points,
)

print(f"Uploaded {len(points)} vectors to Qdrant.")

Uploaded 295 vectors to Qdrant.


## Validate Qdrant Collection

In [27]:
collection_info = client.get_collection(COLLECTION_NAME)

print("Collection:", COLLECTION_NAME)
print("Vectors stored:", collection_info.points_count)

assert collection_info.points_count == len(chunks)

print("Qdrant validation passed.")

Collection: financial_policies
Vectors stored: 295
Qdrant validation passed.
